# Ozone groups of double extension regular algebras of type (14641)
**James Gómez · Helbert Venegas**  
Computational verification companion — working audit, 22 November 2026.

This notebook performs **exact consistency tests** against the sample table and the formulae stated in the authors’ manuscripts. It does **not** by itself prove that a proposed subgroup exhausts the full ozone group, nor does it reproduce all Gröbner-basis central calculations. The separation between theorem-based tests and independent relation-based tests is intentional.

In Google Colab, upload `ozone_table.csv` to the same workspace as the notebook (or clone the repository). No floating-point arithmetic is used.

In [ ]:
import csv, re, math
from pathlib import Path
from collections import Counter
import sympy as sp
from sympy.matrices.normalforms import smith_normal_form
from sympy.polys.domains import ZZ

source = Path('ozone_table.csv')
if not source.exists():
    source = Path('/mnt/data/ozone_verification/ozone_table.csv')
assert source.exists(), 'Upload ozone_table.csv next to this notebook.'
with source.open(encoding='utf-8', newline='') as f:
    table = list(csv.DictReader(f))
for r in table:
    for col in ('orden','rango','cociente'):
        r[col] = int(r[col]) if r[col] else None
print('Rows:',len(table),'Families:',sorted(set(r['familia'] for r in table)))

## 1. Exact group orders and central ranks
Check elementary invariant-factor arithmetic, the rank identity, divisibility, and the strict inequality. These are necessary consistency conditions, not proofs of group completeness.

In [ ]:
def invariant_factors_from_latex(s):
    """Parse the finite abelian groups occurring in the published sample table."""
    s = s.replace('\\mathbb Z','Z').replace('\\times','*').replace('\\left','').replace('\\right','')
    s = s.replace(' ','')
    out = []
    # Parenthesized terms such as (Z/2)^{3}, (Z/4)^{2}
    pattern = r'\(Z/(\d+)\)\^\{?(\d+)\}?'
    for m in list(re.finditer(pattern,s)):
        out.extend([int(m.group(1))]*int(m.group(2)))
    s = re.sub(pattern, '', s)
    out += [int(x) for x in re.findall(r'Z/(\d+)',s)]
    if not out:
        raise ValueError('Unrecognized group '+repr(s))
    return sorted(out)

issues=[]
for k,r in enumerate(table,1):
    factors=invariant_factors_from_latex(r['grupo'])
    if math.prod(factors)!=r['orden']:
        issues.append((k,'group order',r))
    if r['rango'] is not None:
        if r['rango']!=r['orden']*r['cociente']:
            issues.append((k,'rank/quotient',r))
        if math.isqrt(r['rango'])**2!=r['rango']:
            issues.append((k,'rank not square',r))
        if r['rango']%r['orden']!=0:
            issues.append((k,'nondivisible order',r))
        if r['rango']<=r['orden']:
            issues.append((k,'non-strict bound',r))
print('Arithmetic inconsistencies:',issues)
assert not issues


## 2. Root-of-unity arithmetic (exact)
We represent roots of unity as exact SymPy expressions. `root_order` searches for the minimal positive exponent; it is suitable for the sample values (orders ≤ 100), not as a general PI-decision procedure.

In [ ]:
I=sp.I
omega=(-1+sp.sqrt(3)*I)/2

def root_order(x, bound=200):
    x=sp.simplify(x)
    for k in range(1,bound+1):
        if sp.simplify(sp.expand_complex(x**k)-1)==0:
            return k
    return None

def lcm(*args):
    return math.lcm(*args)

assert root_order(I)==4 and root_order(omega)==3
assert root_order((1+I)/(1-I))==4
print('Exact arithmetic operational')

## 3. Families K and L: independent application of manuscript PI-degree formulas
From the newer monomial manuscript:

\(\mathrm{PIdeg}\,K(q,f,h)=2\operatorname{lcm}(2,\mathrm{ord}(q),\mathrm{ord}(f),\mathrm{ord}(h))\),
\(\mathrm{PIdeg}\,L(q,f,h)=2\operatorname{lcm}(2,\mathrm{ord}(q),\mathrm{ord}(f),\mathrm{ord}(h^2))\).

In both cases \(|\mathrm{Oz}|=(\mathrm{PIdeg})^2/2\).

In [ ]:
def qfh_of_row(r):
    def parse_scalar(t):
        return {'-1':sp.Integer(-1),'1':sp.Integer(1),'i':I}.get(t)
    params=r['parametros'].replace('$','').replace(' ','')
    q=parse_scalar(re.search(r'q=([^,]+)',params).group(1))
    f=parse_scalar(re.search(r'f=([^,]+)',params).group(1))
    h=parse_scalar(r['h'].replace('$',''))
    return q,f,h

kl_issues=[]
for r in table:
    if r['familia'] not in ('K','L'): continue
    q,f,h=qfh_of_row(r)
    orders=[root_order(x) for x in (q,f,h if r['familia']=='K' else h**2)]
    degree=2*lcm(2,*orders)
    if r['rango'] != degree**2 or r['orden']*2 !=degree**2:
        kl_issues.append((r,degree))
print('K/L examples:',sum(r['familia'] in ('K','L') for r in table))
print('K/L formula mismatches:',len(kl_issues))
assert not kl_issues

## 4. One-parameter family B
The manuscript predicts \(n=\operatorname{ord}(h^4)\), group order \(8n^2\), rank \(64n^2\), and invariant factors \(C_{2n}	imes C_{4n}\) for odd \(n\), or \(C_2	imes C_{2n}^2\) for even \(n\).

In [ ]:
b_issues=[]
for r in table:
    if r['familia']!='B': continue
    h={'-1':-sp.Integer(1),'1':sp.Integer(1),'i':I}[r['h'].strip('$')]
    n=root_order(h**4)
    expected=sorted([2*n,4*n] if n%2 else [2,2*n,2*n])
    if (r['orden'],r['rango'],invariant_factors_from_latex(r['grupo'])) != (8*n*n,64*n*n,expected):
        b_issues.append(r)
print('B discrepancies:',b_issues)
assert not b_issues

## 5. Two-parameter monomial families M, N, O, P, W, Z
Use the exact parameters \(u,v\) listed in Table `tab:uv` of the newer manuscript, and set \(m=\operatorname{ord}(u^2)\), \(m'=\operatorname{ord}(v^2)\). Check the order and PI degree formula, without using the reported center. We take \(s^2=f\), choosing the displayed square root in the sample rows; changing its sign does not change the order/rank predictions.

In [ ]:
def two_param_uv(r):
    fam=r['familia']
    h=(1+I)/2
    if fam=='N':
        f=(1+I)/2;g=(1-I)/2
        return sp.cancel((f-g)/(f+g)),sp.expand(f*f-g*g)
    if fam in 'MOP':
        f=sp.Integer(-1);s=I
        u=(s+1)/(s-1) if fam=='M' else (1+s)/(1-s)
        v=h*h*(f-1 if fam=='M' else 1-f)
        return sp.simplify(u),sp.simplify(v)
    if fam in 'WZ':
        f=sp.Integer(1);s=sp.Integer(1)
        u=(s-I)/(s+I) if fam=='W' else (I-s)/(I+s)
        v=h*h*(f+1)
        return sp.simplify(u),sp.simplify(v)
    raise ValueError(fam)

two_issues=[];two_details=[]
for r in table:
    if r['familia'] not in 'MNOPWZ':continue
    u,v=two_param_uv(r)
    ou,ov=root_order(u),root_order(v)
    m,mp=root_order(u*u),root_order(v*v)
    if r['familia'] in 'MNP':
        expected_order=8*mp**2
        degree=4*m*mp
    else:
        expected_order=(8 if ou%2==0 else (16 if ov%2==0 else 4))*mp**2
        degree=(2 if ou%2 and ov%2 else 4)*m*mp
    two_details.append((r['familia'],str(u),str(v),m,mp,expected_order,degree))
    if r['orden']!=expected_order or r['rango']!=degree**2:
        two_issues.append((r,expected_order,degree))
print('Family, u, v, m, m-prime, predicted ozone order, predicted PI degree')
for line in two_details: print(line)
print('Discrepancies:',two_issues)
assert not two_issues

## 6. Smith normal form: actual invariant factors
These are the integer relation matrices exhibited for the one-parameter families in the manuscript, not an independent construction of the center. The Smith form is computed exactly over the integers.

In [ ]:
def snf_nontrivial(A):
    S=smith_normal_form(sp.Matrix(A),domain=ZZ)
    return sorted(abs(int(S[i,i])) for i in range(min(S.shape)) if abs(int(S[i,i]))>1)

for n in (1,2,3,4,5,6):
    MB=[[2*n,0,n],[0,2*n,n],[0,0,2]]
    ME=[[2*n,0,0],[0,2*n,n],[0,0,2]]
    expected=sorted([2*n,4*n] if n%2 else [2,2*n,2*n])
    # SNF of the B/E presentations as printed
    assert snf_nontrivial(MB)==expected, (n,snf_nontrivial(MB),expected)
    assert snf_nontrivial(ME)==expected, (n,snf_nontrivial(ME),expected)
print('Smith normal form checks for B, E and J: passed for n = 1,...,6')

## 7. Other one-parameter families, and the K/L Calabi–Yau criterion
The sample values of \(h\) satisfy \(n=\operatorname{ord}(4h^4)\) for E/J and \(n=\operatorname{ord}(16h^4)\) for S/T/U. Their invariant factors are tested against Proposition `prop:oneparam-structure`.

For K and L, the manuscript gives **K never CY**, and **L CY iff \(h^2=q\), \(f=-q^{-1}\)**. These are checks against the stated criterion, rather than an independent Nakayama automorphism computation.

In [ ]:
one_issues=[]
for r in table:
    fam=r['familia']
    if fam not in 'EJSTU': continue
    if fam in 'EJ':
        hsquared = I/2 if fam=='E' else sp.Rational(1,2)*sp.Integer(1)
        n=root_order(4*hsquared**2)
        predicted_order=8*n*n
        predicted_rank=64*n*n
        factors=sorted([2*n,4*n] if n%2 else [2,2*n,2*n])
    else:
        h = {'1/2':sp.Rational(1,2), 'i/2':I/2}[r['h'].strip('$')]
        n=root_order(16*h**4)
        predicted_order=(16 if fam=='S' else 8)*n*n
        predicted_rank=64*n*n
        factors=sorted(([2,2,2*n,2*n] if fam=='S' else [2,2*n,2*n]))
    if (r['orden'],r['rango'],invariant_factors_from_latex(r['grupo'])) != (predicted_order,predicted_rank,factors):
        one_issues.append((fam,r))
cy_issues=[]
for r in table:
    if r['familia'] not in ('K','L'):continue
    q,f,h=qfh_of_row(r)
    predicted=(r['familia']=='L' and sp.simplify(h*h-q)==0 and sp.simplify(f+1/q)==0)
    actual=r['CY']=='sí'
    if predicted!=actual:cy_issues.append(r)
print('E,J,S,T,U formula/structure mismatches:',one_issues)
print('K/L Calabi-Yau mismatches:',cy_issues)
assert not one_issues and not cy_issues

## 8. Important limitations and next certificates
**Not verified here:** the exhaustive system of eight polynomial unknowns in the older manuscript, the full centers of the nonmonomial families, exact CY criteria outside K/L, nor the starred rows without proven hsop.

**Before describing the repository as a complete verification:** implement a PBW normal-form reducer for each set of Zhang–Zhang relations; check the proposed automorphism action on all six defining relations, fix a proven generating set for the center, and establish completeness using the mathematical normality/Galois argument or certified zero-dimensional Gröbner computation. For finite-field checks, include the prime, exact characteristic, reduction of parameters and lifting rationale: counting solutions modulo one prime alone does not certify characteristic-zero group structure.

The older draft contains a claim that all monomial ozone groups arise solely as characters of one fixed monomial grading. The newer manuscript explicitly includes permutation cosets in several families, so this assertion must not be reused without its stronger hypotheses.

## 9. Coverage of the 26 Zhang–Zhang families
The catalog includes **all 26 families**, but a row in the catalog is **not** a computational proof. The older draft calls the general PI criteria in C, F, I observational; the new general monomial theorems address fourteen families. A, H, X and Y are excluded from the PI locus in the classification draft. Family R is covered there by an external result attributed to Zhang–Zhang.

The current notebook separates (a) proven general formulas applied to sampled parameters, (b) exact degree-two algebra-relation tests, and (c) outstanding center-fixed/exhaustiveness certificates.

In [ ]:
import csv
from collections import Counter
with open('family_catalog.csv',encoding='utf8',newline='') as _f:
    catalog=list(csv.DictReader(_f))
assert len(catalog)==26 and set(r['family'] for r in catalog)==set('ABCDEFGHIJKLMNOPQRSTUVWXYZ')
print('Family status:',dict(Counter(r['method'] for r in catalog)))
for row in catalog: print(row['family'],row['method'],':',row['PI_condition_from_draft'])

## 10. New independent certificate: degree-two defining relations

This is a **new exact symbolic check**, not just a comparison of formulas or group orders. For each of the fourteen monomial families, `degree2_certificates.py` constructs all six quadratic relations in the monomial basis from Table `tab:mono`, reduces their images to a degree-two PBW basis, and checks whether specified block-linear maps preserve those relations. The code uses exact SymPy arithmetic, never approximate floating point.

It performs a small search over 16 sign automorphisms and four pure permutation patterns, each at a particular PI specialization. **Passing this test means that an invertible map preserves the defining ideal, not that it fixes the center.** Some valid algebra automorphisms shown here are *not* ozone automorphisms.

In [ ]:
from degree2_certificates import family_data,verify_linear_action,diagonal_sign_solutions,permutation_candidates
import sympy as sp
monomial_families=sorted(set('BEJKLM NOPSTUWZ'.replace(' ','')))
relation_results={}
for fam in monomial_families:
    d=family_data(fam)
    ident=verify_linear_action(d,sp.eye(2),sp.eye(2))
    assert len(ident)==6 and all(not r for r in ident),fam
    signs=diagonal_sign_solutions(d)
    patterns=permutation_candidates(d)
    relation_results[fam]={'identity':True,'sign_automorphisms':len(signs),'pure_permutations':patterns}
    print(fam, 'signed algebra automorphisms:',len(signs),'pure permutations:',patterns)
assert len(relation_results)==14

## 11. General symbolic polynomial system in eight unknowns

This produces the **relation-preservation portion (i)** of Corollary `cor:ozsist`. It is applicable to any of the fourteen monomial families. Center fixing (ii) requires a proven complete generating set for the center. Invertibility (iii) is 
`det(X)*det(Y) != 0`.

The eight variables are the entries of the two 2-by-2 matrices. The algorithm reduces the transformed relations to the ten-dimensional degree-two PBW normal form and extracts each coefficient as an exact polynomial equation.

In [ ]:
a,b,c,d,u,v,w,z=sp.symbols('a b c d u v w z')
X=sp.Matrix([[a,b],[c,d]]);Y=sp.Matrix([[u,v],[w,z]])
example='L'
residuals=verify_linear_action(family_data(example),X,Y)
polynomials=sorted(set(sp.expand(val) for r in residuals for val in r.values()),key=str)
assert all(sp.simplify(p.subs({a:1,b:0,c:0,d:1,u:1,v:0,w:0,z:1}))==0 for p in polynomials)
print('Family:',example,'quadratic relation equations:',len(polynomials))
for expr in polynomials:print(sp.Eq(expr,0))
print('Invertibility: (a*d-b*c)*(u*z-v*w) != 0')

## 12. A nonmonomial PI-degree check: family G at h=1

In the older manuscript the associated graded quantum affine algebra has skew exponent matrix
\[
E=\begin{pmatrix}0&0&-1&-1\\0&0&-2&0\\1&2&0&-1\\1&0&1&0\end{pmatrix}.
\]
Its Smith factors are `1,1,2,2`. For a primitive n-th root p and **h=1**, the draft proves the PI degree of G is `n*n/gcd(n,2)`; this test checks the lattice computation and the sampled ranks for n=3 and n=4. It is not a full relation-level ozone-group proof for G.

In [ ]:
from sympy.matrices.normalforms import smith_normal_form
from sympy.polys.domains import ZZ
E=sp.Matrix([[0,0,-1,-1],[0,0,-2,0],[1,2,0,-1],[1,0,1,0]])
S=smith_normal_form(E,domain=ZZ)
assert sorted(abs(int(S[i,i])) for i in range(4))==[1,1,2,2]
for n in (3,4):
    degree=n*n//math.gcd(n,2)
    matching=[r for r in table if r['familia']=='G' and r['h']=='$1$' and (('p=\omega' in r['parametros']) if n==3 else ('p=i' in r['parametros']))]
    assert len(matching)==1,(n,matching)
    assert matching[0]['rango']==degree**2,(n,matching[0])
    print('G, h=1, ord(p)=',n,': PI degree=',degree, ', rank=',degree**2)

## 13. Release criteria

- **Complete at present:** transcription and arithmetic audits of 73 published rows; application of manuscript formulas in the monomial sample rows; 14 exact monomial degree-two relation engines; generated polynomial constraint equations; 26-family catalog.
- **Not complete:** center-generator certificates and full eight-variable solving for the eight nonmonomial families C,D,F,G,I,Q,R,V; general PI classification of C,F,I in the older manuscript; no uniform ozone theorem for the nonmonomial families; characteristic-zero proofs of group completeness; starred center computations.
- **Important:** the generic relation solver must not be described to editors as a completed center-fixing or ozone-classification proof until those additional conditions are implemented and verified.

## 14. Universal Zhang–Zhang quadratic relation engine (all 26 families)

`general_quadratic.py` accepts the defining coefficients \(Q=(q_{12},q_{11})\), \(P=(p_{12},p_{11})\), and the \(4\times4\) matrix \(\Sigma\) from the source classification. Its rows refer to \((y_1x_1,y_1x_2,y_2x_1,y_2x_2)\), and its columns to \((x_1y_1,x_2y_1,x_1y_2,x_2y_2)\). It checks the preservation of all six relations using a PBW normal form, even for Jordan planes and nonmonomial mixed relations.

**The engine supports A–Z structurally; the actual source matrices for C, D, F, G, I, Q, R, V are not yet transcribed.** They must be entered from the Zhang–Zhang classification, checked against the manuscript, and only then run as a valid certificate.

In [ ]:
from general_quadratic import Datum,verify_general,from_monomial_coordinate
for fam in ('B','L'):
    datum=from_monomial_coordinate(family_data(fam))
    generic=verify_general(datum,sp.eye(2),sp.eye(2))
    assert len(generic)==6 and all(not residual for residual in generic)
    assert datum.sigma.shape==(4,4)
    print(fam,'generic Sigma engine validated against monomial model')
# For a new family supply Datum(q12,q11,p12,p11,Sigma), with Sigma ordered as above.

## 15. Boundaries of the present release

The phrase **“all PI families”** now means that every family is tracked in the catalog and there is a generic relation verification method ready for every family. It **does not mean** every family has an independently certified ozone group: an exact defining matrix, complete central generators, and a completeness proof are needed before making that statement. The family-wide results on C, F and I remain at the status indicated in the manuscripts.

## 16. v3 — Source-traceable quadratic presentations and exact regression report
The monomial changes of basis are transcribed from `tab:mono` (Gómez–Venegas, 18 Nov 2026). We reconstruct each original-coordinate coefficient matrix by exact conjugation, and compare two independent degree-two reduction routines on 20 candidate actions per family. **These reconstructed matrices are not independent transcriptions from Zhang–Zhang.**

In [ ]:
from source_monomial import FAMILIES, valid_specialization, original_datum_from_monomial, check_rebased_planes
from run_certificates import run
cert = run()
assert cert["family_total"] == 26
assert cert["sample_rows"] == 73
assert cert["quadratic_action_crosschecks"] == 280
print("Exact checks:", cert["quadratic_action_crosschecks"], "| Families:", cert["family_total"])

## 17. Representative parameters and coefficient matrices
All 14 monomial families have a valid chosen representative and an exact quadratic Sigma reconstructed in the original coordinates. This step validates the basis conversion, not the original Zhang–Zhang table independently.

In [ ]:
for family in FAMILIES:
    datum = original_datum_from_monomial(family)
    p = valid_specialization(family)
    assert check_rebased_planes(family)
    print(family, "parameters:", p, "nonzero Sigma coefficients:", sum(v != 0 for v in datum.sigma))

## 18. Required work before an exhaustive ozone certificate
For each remaining family, import the **independently verified** Sigma from Zhang–Zhang, compute a *proven* set of central generators, impose `sigma(c)=c` and determinant nonvanishing, and solve the resulting zero-dimensional system over characteristic zero. The finite-field computations in the earlier draft are evidence, not automatic characteristic-zero certification. The starred rows remain conditional. See `VERIFICATION_STATUS.md` and `certificate_report.json`.

## Theorem-level certificates: G(p,f,1), and D necessary conditions

See `DG_THEOREM_NOTE.tex` for the written proof. The exact checks below enumerate finite cyclic exponent data; the proof of exhaustivity for G uses the full center theorem in the manuscript. For D the computed set is only an upper bound.

In [ ]:
from dg_exact import run, g_group, expected_factors, d_necessary
run()

## v5 — Verified quotient-conic dynamics for Q and V

This section checks **only** the normal-localization characteristic polynomials, conic invariance and Vieta orders. It does **not** calculate the ozone groups. The R-family needs its own quadratic-normal localization.

In [ ]:
from qrv_dynamics import certificate
for family in ("Q", "V"):
    print(certificate(family))

## Exact ozone analysis for Q(1) and V(1) (v6)
The Q(1) claim is proved using an additional central element of bidegree (1,2) and normal conjugation by x1. The V(1) calculation is only an upper bound.


In [ ]:
from qrv_ozone_h1 import Q_certificate, V_certificate
print(Q_certificate())
print(V_certificate())


## v7 — Completed ozone groups for Q(1), R(1) and V(1)

Each exact proof combines (a) an upper bound from necessary central equations and the quadratic relations, with (b) a lower bound from an explicit *normal-inner automorphism* of order four. Hence each specialized ozone group is cyclic of order four. See `QRV_H1_COMPLETE.tex` for the mathematical proofs, and `INFORME_AVANCE.md` for the unproved generic twists. The R presentation is independently sourced and must be checked against original ZZ09 before submission.

In [ ]:
from qrv_ozone_h1 import Q_certificate
from r1_normal_certificate import certificate as R_certificate
from v1_normal_certificate import certificate as V_certificate
Q1, R1, V1 = Q_certificate(), R_certificate(), V_certificate()
assert Q1['ozone_group'] == 'C4'
assert R1['candidates_after_new_center'] == 4
assert V1['upper_bound'] == 4
print('Q(1):', Q1['ozone_group'])
print('R(1):', R1['result'], '| normal =', R1['normal_element'])
print('V(1):', V1['result'], '| normal =', V1['normal_element'])

## Version 8 — 26-family audit (scope-controlled)

This is a **per-family audit**, not an exhaustive theorem for all PI parameters. The complete report differentiates numerical table checks, exact quadratic relation checks, and conditional or specialized ozone proofs. Generic twists remain open for D, G, Q, R, V; generic PI/ozone classification is pending for C, F, I.

In [ ]:
from audit_all_families import main
v8 = main()
assert v8["source_cases"] == 26
assert v8["sample_arithmetic_passed"] == 73
for family, status in v8["families"].items():
    print(family, status["verified"])